In [10]:
from __future__ import annotations
from pathlib import Path
from typing import Union
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
import csv
import os
import fastparquet
import tqdm

In [11]:


def load_training_dataframe(parquet_root: Union[str, Path]) -> pd.DataFrame:
    """Load and prepare parquet files for model training.

    Parameters
    ----------
    parquet_root:
        Directory containing parquet files. All parquet files found recursively
        under this directory are concatenated into a single DataFrame.

    Returns
    -------
    pd.DataFrame
        Data ready for XGBoost. Rows with missing values are dropped and
        columns are converted to integer or boolean types where appropriate.
    """

    parquet_root = Path(parquet_root)
    files = sorted(parquet_root.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found under {parquet_root}")

    # Read all parquet files and concatenate
    frames = [pd.read_parquet(f, engine="pyarrow") for f in files]
    df = pd.concat(frames, ignore_index=True)

    # Drop any rows containing NA values
    df = df.dropna().reset_index(drop=True)

    # Convert object columns to numeric or categorical codes
    obj_cols = df.select_dtypes(include="object").columns 
    obj_cols = obj_cols.drop("match_id")
    for col in obj_cols:
        lower = df[col].str.lower()
        if set(lower.unique()) <= {"true", "false"}:
            df[col] = lower == "true"
        else:
            df[col] = df[col].astype("category").cat.codes

    # Convert numeric columns to int/bool where appropriate; keep time as float
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        series = df[col]
        # Preserve time columns as float (avoid 0/1 -> bool)
        if str(col).lower() in {"time", "elapsed_time"}:
            df[col] = pd.to_numeric(series, errors="coerce").astype(float)
            continue
        if pd.api.types.is_float_dtype(series) and np.allclose(series, series.astype(int)):
            series = series.astype(int)
        # Only convert to bool if truly binary (ignoring NaN)
        if set(pd.Series(series).dropna().unique()) <= {0, 1}:
            series = series.astype(bool)
        else:
            series = pd.to_numeric(series, downcast="integer")
        df[col] = series

    return df

In [12]:
def _rolling_ratio(df: pd.DataFrame, num: str, den: str, window: int) -> pd.Series:
    """Compute rolling ratio of cumulative columns num/den over a window
    based on last `window` rows within each match.

    This is kept for simple row-based windows, but for event-based windows
    (e.g., last N serves), prefer _event_rolling_ratio.
    """
    num_cum = df[num]
    den_cum = df[den]
    num_win = num_cum - num_cum.groupby(df["match_id"]).shift(window).fillna(0)
    den_win = den_cum - den_cum.groupby(df["match_id"]).shift(window).fillna(0)
    return _safe_div(num_win, den_win)


def _event_rolling_ratio(df: pd.DataFrame, num_cum_col: str, den_cum_col: str, event_cum_col: str, window: int) -> pd.Series:
    """Event-based rolling ratio over the last `window` events.

    Parameters
    ----------
    num_cum_col : str
        Name of cumulative numerator column.
    den_cum_col : str
        Name of cumulative denominator column.
    event_cum_col : str
        Name of cumulative event counter (increments by 1 on each event).
    window : int
        Window size in number of events.
    """
    out = pd.Series(index=df.index, dtype=float)
    for mid, g in df.groupby("match_id", sort=False):
        e = g[event_cum_col].astype(float)
        # Rows where an event occurs (counter increments)
        ev_mask = (e.diff().fillna(e) > 0)
        if not ev_mask.any():
            out.loc[g.index] = 0.0
            continue
        ev = g.loc[ev_mask, [num_cum_col, den_cum_col]].copy()
        num_inc = ev[num_cum_col].diff().fillna(ev[num_cum_col])
        den_inc = ev[den_cum_col].diff().fillna(ev[den_cum_col])
        num_roll = num_inc.rolling(window, min_periods=1).sum()
        den_roll = den_inc.rolling(window, min_periods=1).sum()
        ratio_ev = _safe_div(num_roll, den_roll)
        # Map back to full index, forward-filling between events
        out.loc[g.index] = ratio_ev.reindex(g.index).ffill().fillna(0.0).to_numpy()
    return out


def _event_rolling_sum(df: pd.DataFrame, cum_col: str, event_cum_col: str, window: int) -> pd.Series:
    """Event-based rolling sum of a cumulative column over last `window` events.
    """
    out = pd.Series(index=df.index, dtype=float)
    for mid, g in df.groupby("match_id", sort=False):
        e = g[event_cum_col].astype(float)
        ev_mask = (e.diff().fillna(e) > 0)
        if not ev_mask.any():
            out.loc[g.index] = 0.0
            continue
        ev = g.loc[ev_mask, [cum_col]].copy()
        inc = ev[cum_col].diff().fillna(ev[cum_col])
        roll = inc.rolling(window, min_periods=1).sum()
        out.loc[g.index] = roll.reindex(g.index).ffill().fillna(0.0).to_numpy()
    return out



In [13]:
data = load_training_dataframe("data/matches")

In [14]:
data.to_parquet("data/processed/data_processed.parquet", index=False)

In [15]:
def pair_match_point_files(file_names):
    file_set = set(file_names)
    pairs = []
    for name in file_names:
        if "matches" in name:
            points_name = name.replace("matches", "points")
            if points_name in file_set:
                pairs.append([name, points_name])
    return pairs

In [16]:
files = pair_match_point_files(os.listdir('data/raw data'))

In [17]:
def _safe_div(a: pd.Series, b: pd.Series) -> pd.Series:
    """Safely divide two Series avoiding division by zero.

    Parameters
    ----------
    a, b : pd.Series
        Numerator and denominator.
    Returns
    -------
    pd.Series
        Series of a / b where b is non-zero, otherwise 0.
    """
    a = a.astype(float)
    b = b.astype(float)
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def _is_game_point_for(pts_for: pd.Series, pts_against: pd.Series, is_tiebreak: pd.Series) -> pd.Series:
    """Return True if the upcoming point is a game point for the perspective player."""
    # Regular games: win when reaching at least 4 points and leading by 2.
    is_tb = is_tiebreak.astype(bool)
    cond1 = (pts_for == 3) & (pts_against <= 2) & (~is_tb)
    cond2 = (pts_for >= 3) & (pts_against >= 3) & (pts_for == pts_against + 1) & (~is_tb)
    return cond1 | cond2


def _is_game_point_against(pts_for: pd.Series, pts_against: pd.Series, is_tiebreak: pd.Series) -> pd.Series:
    """Game point for the opponent (simply flip arguments)."""
    return _is_game_point_for(pts_against, pts_for, is_tiebreak)



In [18]:
def build_match_state_panel(input_data: pd.DataFrame, best_of_default: int = 5, ma_window: int = 60) -> pd.DataFrame:
    """Construct a panel with match state before each point for both perspectives."""
    df = input_data.copy()

    needed = ["match_id", "SetNo", "GameNo", "PointNumber", "PointServer"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    if "PointWinner" in df.columns:
        p1_won_point = (df["PointWinner"] == 1).astype(int)
        p2_won_point = (df["PointWinner"] == 2).astype(int)
    elif {"P1PointsWon", "P2PointsWon"}.issubset(df.columns):
        df = df.sort_values(["match_id", "SetNo", "GameNo", "PointNumber"]).copy()
        df["p1_cum_prev"] = df.groupby("match_id")["P1PointsWon"].shift(1).fillna(0)
        df["p2_cum_prev"] = df.groupby("match_id")["P2PointsWon"].shift(1).fillna(0)
        p1_won_point = (df["P1PointsWon"] > df["p1_cum_prev"]).astype(int)
        p2_won_point = (df["P2PointsWon"] > df["p2_cum_prev"]).astype(int)
    else:
        raise ValueError(
            "Need either PointWinner or (P1PointsWon, P2PointsWon) to derive point winners."
        )

    df = df.sort_values(["match_id", "SetNo", "GameNo", "PointNumber"]).copy()
    df["point_idx"] = df.groupby("match_id").cumcount() + 1

    # Serve counts and speeds
    if "Speed_KMH" in df.columns:
        df["Speed_KMH"] = df["Speed_KMH"].fillna(0).astype(float)
    else:
        df["Speed_KMH"] = 0.0
    for side in (1, 2):
        srv_mask = (df["PointServer"] == side).astype(int)
        df[f"p{side}_sv_pts"] = srv_mask.groupby(df["match_id"]).cumsum().shift(1).fillna(0)
        df[f"p{side}_srv_speed_cum"] = (
            (df["Speed_KMH"] * srv_mask)
            .groupby(df["match_id"])
            .cumsum()
            .shift(1)
            .fillna(0)
        )
        df[f"p{side}_avg_srv_speed"] = _safe_div(
            df[f"p{side}_srv_speed_cum"], df[f"p{side}_sv_pts"]
        )

    # Match level cumulative points before current point
    df["ttl_p1"] = p1_won_point.groupby(df["match_id"]).cumsum().shift(1).fillna(0).astype(int)
    df["ttl_p2"] = p2_won_point.groupby(df["match_id"]).cumsum().shift(1).fillna(0).astype(int)

    # Optional serve/return counters and rates
    srv_cols = {
        "P1FirstSrvIn",
        "P1FirstSrvWon",
        "P1SecondSrvIn",
        "P1SecondSrvWon",
        "P1DoubleFault",
        "P2FirstSrvIn",
        "P2FirstSrvWon",
        "P2SecondSrvIn",
        "P2SecondSrvWon",
        "P2DoubleFault",
    }
    have_srv = srv_cols.issubset(df.columns)
    if have_srv:
        for side in (1, 2):
            df[f"p{side}_fs_in_cum"] = (
                df[f"P{side}FirstSrvIn"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
            )
            df[f"p{side}_fs_won_cum"] = (
                df[f"P{side}FirstSrvWon"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
            )
            df[f"p{side}_ss_in_cum"] = (
                df[f"P{side}SecondSrvIn"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
            )
            df[f"p{side}_ss_won_cum"] = (
                df[f"P{side}SecondSrvWon"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
            )
            df[f"p{side}_df_cum"] = (
                df[f"P{side}DoubleFault"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
            )
    else:
        for side in (1, 2):
            df[f"p{side}_fs_in_cum"] = df[f"p{side}_fs_won_cum"] = 0.0
            df[f"p{side}_ss_in_cum"] = df[f"p{side}_ss_won_cum"] = 0.0
            df[f"p{side}_df_cum"] = 0.0

    if {"P1Ace", "P2Ace"}.issubset(df.columns):
        df["p1_aces_cum"] = df["P1Ace"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
        df["p2_aces_cum"] = df["P2Ace"].groupby(df["match_id"]).cumsum().shift(1).fillna(0)
    else:
        df["p1_aces_cum"] = df["p2_aces_cum"] = 0.0

    # Derived serve/return rates
    for side in (1, 2):
        df[f"p{side}_fs_in_pct"] = _safe_div(df[f"p{side}_fs_in_cum"], df[f"p{side}_sv_pts"])
        df[f"p{side}_ss_att_cum"] = df[f"p{side}_ss_in_cum"] + df[f"p{side}_df_cum"]
        df[f"p{side}_ss_in_pct"] = _safe_div(df[f"p{side}_ss_in_cum"], df[f"p{side}_ss_att_cum"])
        df[f"p{side}_fs_win_pct"] = _safe_div(df[f"p{side}_fs_won_cum"], df[f"p{side}_fs_in_cum"])
        df[f"p{side}_ss_win_pct"] = _safe_div(df[f"p{side}_ss_won_cum"], df[f"p{side}_ss_in_cum"])
        recv_mask = (df["PointServer"] != side).astype(int)
        won = (p1_won_point if side == 1 else p2_won_point) * recv_mask
        df[f"p{side}_ret_won_cum"] = won.groupby(df["match_id"]).cumsum().shift(1).fillna(0)
        df[f"p{side}_ret_pts_cum"] = recv_mask.groupby(df["match_id"]).cumsum().shift(1).fillna(0)
        df[f"p{side}_ret_win_pct"] = _safe_div(
            df[f"p{side}_ret_won_cum"], df[f"p{side}_ret_pts_cum"]
        )
        # Match-wide ace rate (constant across the match)
        srv_mask_side = (df["PointServer"] == side).astype(int)
        sv_total = srv_mask_side.groupby(df["match_id"]).transform("sum")
        if {"P1Ace", "P2Ace"}.issubset(df.columns):
            aces_total = df[f"P{side}Ace"].groupby(df["match_id"]).transform("sum")
        else:
            aces_total = 0.0
        df[f"p{side}_ace_rate"] = _safe_div(aces_total, sv_total)

        # Event-based rolling window stats (last `ma_window` events)
        df[f"p{side}_avg_srv_speed_l60"] = _event_rolling_ratio(
            df, f"p{side}_srv_speed_cum", f"p{side}_sv_pts", f"p{side}_sv_pts", ma_window
        )
        df[f"p{side}_fs_in_pct_l60"] = _event_rolling_ratio(
            df, f"p{side}_fs_in_cum", f"p{side}_sv_pts", f"p{side}_sv_pts", ma_window
        )
        df[f"p{side}_ss_in_pct_l60"] = _event_rolling_ratio(
            df, f"p{side}_ss_in_cum", f"p{side}_ss_att_cum", f"p{side}_ss_att_cum", ma_window
        )
        df[f"p{side}_fs_win_pct_l60"] = _event_rolling_ratio(
            df, f"p{side}_fs_won_cum", f"p{side}_fs_in_cum", f"p{side}_fs_in_cum", ma_window
        )
        df[f"p{side}_ss_win_pct_l60"] = _event_rolling_ratio(
            df, f"p{side}_ss_won_cum", f"p{side}_ss_in_cum", f"p{side}_ss_in_cum", ma_window
        )
        df[f"p{side}_ret_win_pct_l60"] = _event_rolling_ratio(
            df, f"p{side}_ret_won_cum", f"p{side}_ret_pts_cum", f"p{side}_ret_pts_cum", ma_window
        )
        df[f"p{side}_aces_l60"] = _event_rolling_sum(
            df, f"p{side}_aces_cum", f"p{side}_sv_pts", ma_window
        )
        df[f"p{side}_ace_rate_l60"] = _event_rolling_ratio(
            df, f"p{side}_aces_cum", f"p{side}_sv_pts", f"p{side}_sv_pts", ma_window
        )

    # Time since match start
    if "Time" in df.columns:
        df["Time"] = pd.to_datetime(df["Time"])
        df["elapsed_time"] = df.groupby("match_id")["Time"].transform(lambda s: (s - s.iloc[0]).dt.total_seconds())
    else:
        df["elapsed_time"] = np.nan
    df["elapsed_time"] = df.groupby("match_id")["elapsed_time"].shift(1).fillna(0.0)
    df["elapsed_time"] = df["elapsed_time"].astype(float)

    # Game level point counts before current point
    game_change = (
        (df["SetNo"].astype(str) + "-" + df["GameNo"].astype(str))
        .ne((df["SetNo"].astype(str) + "-" + df["GameNo"].astype(str)).shift(1))
    )
    df["game_key"] = game_change.groupby(df["match_id"]).cumsum()

    df["p1_pts_in_game"] = (
        p1_won_point.groupby([df["match_id"], df["game_key"]]).cumsum().shift(1).fillna(0).astype(int)
    )
    df["p2_pts_in_game"] = (
        p2_won_point.groupby([df["match_id"], df["game_key"]]).cumsum().shift(1).fillna(0).astype(int)
    )

    # ------------------------------------------------------------------
    # Games and sets won before current point
    last_point_idx = df.groupby(["match_id", "SetNo", "GameNo"]).tail(1).index
    game_winner = pd.Series(
        index=last_point_idx,
        data=np.where(
            (p1_won_point + p2_won_point).loc[last_point_idx] == 1,
            np.where(p1_won_point.loc[last_point_idx] == 1, 1, 2),
            np.nan,
        ),
        dtype="float",
    )
    games_tbl = df.loc[last_point_idx, ["match_id", "SetNo", "GameNo"]].copy()
    games_tbl["p1_game_win"] = (game_winner == 1).astype(int)
    games_tbl["p2_game_win"] = (game_winner == 2).astype(int)
    games_tbl["p1_games_in_set_cum"] = (
        games_tbl.groupby(["match_id", "SetNo"])["p1_game_win"].cumsum().shift(1).fillna(0).astype(int)
    )
    games_tbl["p2_games_in_set_cum"] = (
        games_tbl.groupby(["match_id", "SetNo"])["p2_game_win"].cumsum().shift(1).fillna(0).astype(int)
    )
    df = df.merge(
        games_tbl[["match_id", "SetNo", "GameNo", "p1_games_in_set_cum", "p2_games_in_set_cum"]],
        on=["match_id", "SetNo", "GameNo"],
        how="left",
    )
    df.rename(
        columns={"p1_games_in_set_cum": "p1_games_before", "p2_games_in_set_cum": "p2_games_before"},
        inplace=True,
    )

    set_last_games = games_tbl.groupby(["match_id", "SetNo"]).tail(1).copy()
    set_last_games["set_winner"] = np.where(
        set_last_games["p1_games_in_set_cum"] > set_last_games["p2_games_in_set_cum"], 1, 2
    )
    set_last_games["p1_set_win"] = (set_last_games["set_winner"] == 1).astype(int)
    set_last_games["p2_set_win"] = (set_last_games["set_winner"] == 2).astype(int)
    set_last_games["p1_sets_before"] = (
        set_last_games.groupby("match_id")["p1_set_win"].cumsum().shift(1).fillna(0).astype(int)
    )
    set_last_games["p2_sets_before"] = (
        set_last_games.groupby("match_id")["p2_set_win"].cumsum().shift(1).fillna(0).astype(int)
    )

    df = (
        df.merge(
            set_last_games[["match_id", "SetNo", "p1_sets_before", "p2_sets_before"]],
            on=["match_id", "SetNo"],
            how="left",
        ).fillna({"p1_sets_before": 0, "p2_sets_before": 0})
    )

    df["is_tiebreak_game"] = (
        (df["p1_games_before"] == 6) & (df["p2_games_before"] == 6)
    ).astype(int)
    df["server_is_p1"] = (df["PointServer"] == 1).astype(int)

    base_cols = ["match_id", "SetNo", "GameNo", "PointNumber", "point_idx", "elapsed_time"]
    p1 = df[base_cols].copy()
    p1["perspective"] = "P1"
    p1["server_is_persp"] = df["server_is_p1"]
    p1["pts_in_game_for"] = df["p1_pts_in_game"]
    p1["pts_in_game_against"] = df["p2_pts_in_game"]
    p1["games_in_set_for"] = df["p1_games_before"]
    p1["games_in_set_against"] = df["p2_games_before"]
    p1["sets_for"] = df["p1_sets_before"]
    p1["sets_against"] = df["p2_sets_before"]
    p1["is_tiebreak"] = df["is_tiebreak_game"]
    p1["ttl_diff"] = df["ttl_p1"] - df["ttl_p2"]
    p1["aces_diff"] = df["p1_aces_cum"] - df["p2_aces_cum"]
    p1["df_diff"] = df["p1_df_cum"] - df["p2_df_cum"]
    p1["avg_srv_speed"] = df["p1_avg_srv_speed"]
    p1["fs_in_pct"] = df["p1_fs_in_pct"]
    p1["ss_in_pct"] = df["p1_ss_in_pct"]
    p1["fs_win_pct"] = df["p1_fs_win_pct"]
    p1["ss_win_pct"] = df["p1_ss_win_pct"]
    p1["ret_win_pct"] = df["p1_ret_win_pct"]
    p1["aces"] = df["p1_aces_cum"]
    p1["ace_rate"] = df["p1_ace_rate"]
    p1["avg_srv_speed_l60"] = df["p1_avg_srv_speed_l60"]
    p1["fs_in_pct_l60"] = df["p1_fs_in_pct_l60"]
    p1["ss_in_pct_l60"] = df["p1_ss_in_pct_l60"]
    p1["fs_win_pct_l60"] = df["p1_fs_win_pct_l60"]
    p1["ss_win_pct_l60"] = df["p1_ss_win_pct_l60"]
    p1["ret_win_pct_l60"] = df["p1_ret_win_pct_l60"]
    p1["aces_l60"] = df["p1_aces_l60"]
    p1["ace_rate_l60"] = df["p1_ace_rate_l60"]

    p2 = df[base_cols].copy()
    p2["perspective"] = "P2"
    p2["server_is_persp"] = 1 - df["server_is_p1"]
    p2["pts_in_game_for"] = df["p2_pts_in_game"]
    p2["pts_in_game_against"] = df["p1_pts_in_game"]
    p2["games_in_set_for"] = df["p2_games_before"]
    p2["games_in_set_against"] = df["p1_games_before"]
    p2["sets_for"] = df["p2_sets_before"]
    p2["sets_against"] = df["p1_sets_before"]
    p2["is_tiebreak"] = df["is_tiebreak_game"]
    p2["ttl_diff"] = -(df["ttl_p1"] - df["ttl_p2"])
    p2["aces_diff"] = -(df["p1_aces_cum"] - df["p2_aces_cum"])
    p2["df_diff"] = -(df["p1_df_cum"] - df["p2_df_cum"])
    p2["avg_srv_speed"] = df["p2_avg_srv_speed"]
    p2["fs_in_pct"] = df["p2_fs_in_pct"]
    p2["ss_in_pct"] = df["p2_ss_in_pct"]
    p2["fs_win_pct"] = df["p2_fs_win_pct"]
    p2["ss_win_pct"] = df["p2_ss_win_pct"]
    p2["ret_win_pct"] = df["p2_ret_win_pct"]
    p2["aces"] = df["p2_aces_cum"]
    p2["ace_rate"] = df["p2_ace_rate"]
    p2["avg_srv_speed_l60"] = df["p2_avg_srv_speed_l60"]
    p2["fs_in_pct_l60"] = df["p2_fs_in_pct_l60"]
    p2["ss_in_pct_l60"] = df["p2_ss_in_pct_l60"]
    p2["fs_win_pct_l60"] = df["p2_fs_win_pct_l60"]
    p2["ss_win_pct_l60"] = df["p2_ss_win_pct_l60"]
    p2["ret_win_pct_l60"] = df["p2_ret_win_pct_l60"]
    p2["aces_l60"] = df["p2_aces_l60"]
    p2["ace_rate_l60"] = df["p2_ace_rate_l60"]

    panel = pd.concat([p1, p2], ignore_index=True)

    panel["is_game_point_for"] = _is_game_point_for(
        panel["pts_in_game_for"],
        panel["pts_in_game_against"],
        panel["is_tiebreak"].astype(bool),
    ).astype(int)
    panel["is_game_point_against"] = _is_game_point_against(
        panel["pts_in_game_for"],
        panel["pts_in_game_against"],
        panel["is_tiebreak"].astype(bool),
    ).astype(int)
    panel["is_break_point"] = (
        (1 - panel["server_is_persp"]).astype(bool)
        & panel["is_game_point_for"].astype(bool)
    ).astype(int)

    panel["best_of"] = best_of_default
    panel["sets_needed_to_win"] = (panel["best_of"] // 2) + 1

    ordered = [
        "match_id",
        "SetNo",
        "GameNo",
        "PointNumber",
        "point_idx",
        "elapsed_time",
        "perspective",
        "server_is_persp",
        "pts_in_game_for",
        "pts_in_game_against",
        "games_in_set_for",
        "games_in_set_against",
        "sets_for",
        "sets_against",
        "best_of",
        "sets_needed_to_win",
        "is_tiebreak",
        "is_game_point_for",
        "is_game_point_against",
        "is_break_point",
        "ttl_diff",
        "aces_diff",
        "df_diff",
        "avg_srv_speed",
        "fs_in_pct",
        "ss_in_pct",
        "fs_win_pct",
        "ss_win_pct",
        "ret_win_pct",
        "aces",
        "ace_rate",
        "avg_srv_speed_l60",
        "fs_in_pct_l60",
        "ss_in_pct_l60",
        "fs_win_pct_l60",
        "ss_win_pct_l60",
        "ret_win_pct_l60",
        "aces_l60",
        "ace_rate_l60",
        "y_match",
    ]

    panel = panel.merge(
        df.groupby('match_id').tail(1)[['match_id','PointWinner']].rename(columns={'PointWinner':'mw'}),
        on='match_id',
        how='left'
    )
    panel['y_match'] = ((panel['perspective'].eq('P1') & panel['mw'].eq(1)) | (panel['perspective'].eq('P2') & panel['mw'].eq(2))).astype(int)
    panel.drop(columns='mw', inplace=True)
    ordered = [c for c in ordered if c in panel.columns]

    return panel[ordered].copy()

In [19]:
for j in tqdm.tqdm(files):
    path = j[1]
    ex = pd.read_csv(f"data/raw data/{path}")
    games = ex.sort_values(['match_id','SetNo','GameNo','PointNumber']).copy()
    for i in (games['match_id'].unique().tolist()):
        a = (games[games['match_id'] == i])
        name = i.replace('-', '_')
        build_match_state_panel(a).to_parquet(f"data/matches/{name}.parquet", index=False)


 63%|██████▎   | 38/60 [05:33<03:18,  9.02s/it]/tmp/ipykernel_1014/955885429.py:3: DtypeWarning: Columns (64) have mixed types. Specify dtype option on import or set low_memory=False.
  ex = pd.read_csv(f"data/raw data/{path}")
 73%|███████▎  | 44/60 [06:32<02:42, 10.18s/it]/tmp/ipykernel_1014/955885429.py:3: DtypeWarning: Columns (64) have mixed types. Specify dtype option on import or set low_memory=False.
  ex = pd.read_csv(f"data/raw data/{path}")
 85%|████████▌ | 51/60 [07:23<01:02,  6.90s/it]/tmp/ipykernel_1014/955885429.py:3: DtypeWarning: Columns (62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  ex = pd.read_csv(f"data/raw data/{path}")
 97%|█████████▋| 58/60 [08:23<00:20, 10.08s/it]/tmp/ipykernel_1014/955885429.py:3: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  ex = pd.read_csv(f"data/raw data/{path}")
100%|██████████| 60/60 [08:41<00:00,  8.70s/it]
